<a href="https://colab.research.google.com/github/rishabhucb/court-keypoints/blob/main/day3_baseline_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q roboflow ultralytics

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/court_keypoints'
DATA_DIR = f'{PROJECT_DIR}/basketball-court-keypoints-1'

In [ ]:
!ls {DATA_DIR}/train/images | head -20

In [ ]:
import glob, os, collections

def source(fname):
    base = os.path.basename(fname).split('.rf.')[0]   # keep the original name
    return base.rsplit('_', 1)[0]                     # drop the frame number at the end

where = collections.defaultdict(set)
for split in ['train', 'valid', 'test']:
    for f in glob.glob(f'{DATA_DIR}/{split}/images/*'):
        where[source(f)].add(split)

print("Unique sources:", len(where))
leaks = {s: sp for s, sp in where.items() if len(sp) > 1}
print("Sources in more than one split:", len(leaks))
for s, sp in list(leaks.items())[:10]:
    print("  ", s, sorted(sp))

In [ ]:
import re, glob, os

rows, odd = [], []
for split in ['train', 'valid', 'test']:
    for f in glob.glob(f'{DATA_DIR}/{split}/images/*'):
        name = os.path.basename(f)
        m = re.search(r'frame_(\d+)', name)
        if m:
            rows.append((int(m.group(1)), split))
        else:
            odd.append((split, name))
rows.sort()

print("Frame range:", rows[0][0], "to", rows[-1][0], "| frames read:", len(rows))
print("Names without 'frame_':", len(odd))
for s, n in odd[:10]:
    print("  ", s, n)

letters = {'train': 'T', 'valid': 'V', 'test': 'X'}
line = ''.join(letters[s] for n, s in rows)
for i in range(0, len(line), 60):
    print(line[i:i+60])

In [ ]:
!pip install -q imagehash
import imagehash
from PIL import Image

def fp(p): return imagehash.phash(Image.open(p))

train_fps = [fp(p) for p in glob.glob(f'{DATA_DIR}/train/images/*')]
for split in ['valid', 'test']:
    imgs = glob.glob(f'{DATA_DIR}/{split}/images/*')
    near = sum(1 for p in imgs if min(fp(p) - t for t in train_fps) <= 8)
    print(f"{split}: {near} of {len(imgs)} images have a near-duplicate in train")

In [ ]:
import glob, os, imagehash
from PIL import Image

# 1. Collect every image with its label file, from all three splits
items = []
for split in ['train', 'valid', 'test']:
    for img in glob.glob(f'{DATA_DIR}/{split}/images/*'):
        lbl = f'{DATA_DIR}/{split}/labels/' + os.path.splitext(os.path.basename(img))[0] + '.txt'
        items.append((img, lbl))
print(len(items), "images")

# 2. Fingerprint each image once
fps = [imagehash.phash(Image.open(img)) for img, _ in items]

# 3. Link near-duplicates into groups
parent = list(range(len(items)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for i in range(len(items)):
    for j in range(i + 1, len(items)):
        if fps[i] - fps[j] <= 8:
            parent[find(i)] = find(j)

groups = {}
for i in range(len(items)):
    groups.setdefault(find(i), []).append(i)
sizes = sorted((len(g) for g in groups.values()), reverse=True)
print("Number of groups:", len(groups))
print("Largest group sizes:", sizes[:15])
print("Groups of size 1:", sizes.count(1))

In [ ]:
import re
import matplotlib.pyplot as plt

big = sorted(groups.values(), key=len, reverse=True)
A, B = big[0], big[1]

def frame_no(i):
    m = re.search(r'frame_(\d+)', os.path.basename(items[i][0]))
    return int(m.group(1)) if m else 0

for name, g in [('A', A), ('B', B)]:
    nums = [frame_no(i) for i in g]
    print(f"Group {name}: {len(g)} images | frames {min(nums)} to {max(nums)}")

fig, ax = plt.subplots(2, 3, figsize=(15, 6))
for r, g in enumerate([A, B]):
    for c, i in enumerate(g[::max(1, len(g)//3)][:3]):
        ax[r][c].imshow(Image.open(items[i][0])); ax[r][c].axis('off')
    ax[r][0].set_title(f'Group {"AB"[r]}')
plt.show()

In [ ]:
import shutil, yaml

DST = f'{PROJECT_DIR}/court_resplit'
shutil.rmtree(DST, ignore_errors=True)          # start clean if you rerun this cell

B_sorted = sorted(B, key=frame_no)
half = len(B_sorted) // 2
assign = {
    'train': [i for g in big if g is not B for i in g],
    'valid': B_sorted[:half],
    'test':  B_sorted[half:],
}

for split, idx in assign.items():
    os.makedirs(f'{DST}/{split}/images', exist_ok=True)
    os.makedirs(f'{DST}/{split}/labels', exist_ok=True)
    for i in idx:
        img, lbl = items[i]
        shutil.copy(img, f'{DST}/{split}/images/')
        shutil.copy(lbl, f'{DST}/{split}/labels/')
    print(split, len(idx), "images")

cfg = yaml.safe_load(open(f'{DATA_DIR}/data.yaml'))
cfg.update(path=DST, train='train/images', val='valid/images', test='test/images',
           flip_idx=[1, 0, 12, 4, 3, 6, 5, 11, 10, 9, 8, 7, 2])
yaml.safe_dump(cfg, open(f'{DST}/data.yaml', 'w'))
print(open(f'{DST}/data.yaml').read())


In [ ]:
!nvidia-smi

In [ ]:
from ultralytics import YOLO

DATA_YAML = f'{PROJECT_DIR}/court_resplit/data.yaml'

model = YOLO('yolo11n-pose.pt')
results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    fliplr=0.5,
    patience=30,
    project=f'{PROJECT_DIR}/runs',
    name='baseline_v1',
)

In [ ]:
best = YOLO(f'{PROJECT_DIR}/runs/baseline_v1/weights/best.pt')
metrics = best.val(data=DATA_YAML, split='test')
print("Pose mAP50:   ", round(metrics.pose.map50, 3))
print("Pose mAP50-95:", round(metrics.pose.map, 3))

In [ ]:
train_metrics = best.val(data=DATA_YAML, split='train')
print("TRAIN pose mAP50:", round(train_metrics.pose.map50, 3))

In [ ]:
import glob, random
test_imgs = glob.glob(f'{PROJECT_DIR}/court_resplit/test/images/*')
for r in best.predict(random.sample(test_imgs, 5), conf=0.25):
    r.show()

In [ ]:
model = YOLO('yolo11n-pose.pt')
results = model.train(
    data=DATA_YAML,
    epochs=300,
    imgsz=640,
    batch=16,
    fliplr=0.5,
    patience=0,            # no early stopping, run all 300 epochs
    project=f'{PROJECT_DIR}/runs',
    name='baseline_v2',
)

In [ ]:
v2 = YOLO(f'{PROJECT_DIR}/runs/baseline_v2/weights/last.pt')

tr = v2.val(data=DATA_YAML, split='train')
te = v2.val(data=DATA_YAML, split='test')
print("TRAIN pose mAP50:", round(tr.pose.map50, 3))
print("TEST  pose mAP50:", round(te.pose.map50, 3), "| TEST box mAP50:", round(te.box.map50, 3))

In [ ]:
import glob, random
for split in ['train', 'test']:
    imgs = glob.glob(f'{PROJECT_DIR}/court_resplit/{split}/images/*')
    print(f"--- {split} ---")
    for r in v2.predict(random.sample(imgs, 2), conf=0.25):
        r.show()

In [ ]:
print(open(f'{PROJECT_DIR}/court_resplit/test/labels/' +
      os.path.basename(glob.glob(f'{PROJECT_DIR}/court_resplit/test/labels/frame_062*')[0])).read())

In [ ]:
import glob, os

fixed_files = fixed_pts = 0
for lbl in glob.glob(f'{PROJECT_DIR}/court_resplit/*/labels/*.txt'):
    out, changed = [], False
    for line in open(lbl):
        v = line.split()
        if not v:
            continue
        cls = v[0]
        cx, cy, w, h = map(float, v[1:5])
        kp = [float(x) for x in v[5:]]

        # clip the box to the image
        x1, y1 = max(cx - w/2, 0), max(cy - h/2, 0)
        x2, y2 = min(cx + w/2, 1), min(cy + h/2, 1)
        box = [(x1 + x2)/2, (y1 + y2)/2, x2 - x1, y2 - y1]
        if [round(b, 6) for b in box] != [round(b, 6) for b in (cx, cy, w, h)]:
            changed = True

        # hide keypoints that fall outside the image
        for i in range(0, len(kp), 3):
            if not (0 <= kp[i] <= 1 and 0 <= kp[i+1] <= 1):
                kp[i:i+3] = [0.0, 0.0, 0.0]
                fixed_pts += 1
                changed = True

        out.append(' '.join([cls] + [f'{x:.6f}' for x in box + kp]))
    if changed:
        fixed_files += 1
        with open(lbl, 'w') as f:
            f.write('\n'.join(out) + '\n')

for c in glob.glob(f'{PROJECT_DIR}/court_resplit/*/labels.cache'):
    os.remove(c)          # force YOLO to re-read the fixed labels

print(fixed_files, "label files fixed |", fixed_pts, "off-screen keypoints hidden")

In [ ]:
model = YOLO('yolo11n-pose.pt')
results = model.train(
    data=DATA_YAML, epochs=300, imgsz=640, batch=16, fliplr=0.5,
    patience=0, project=f'{PROJECT_DIR}/runs', name='baseline_v3',
)

In [ ]:
import shutil, glob, os, re, yaml

SRC = f'{PROJECT_DIR}/court_resplit'
DST = f'{PROJECT_DIR}/court_v4'
shutil.rmtree(DST, ignore_errors=True)

def fno(p):
    m = re.search(r'frame_(\d+)', os.path.basename(p))
    return int(m.group(1)) if m else 0

other_end = sorted(glob.glob(f'{SRC}/test/images/*'), key=fno)   # 27 frames, in time order
half, gap = len(other_end) // 2, 3

plan = {
    'train': glob.glob(f'{SRC}/train/images/*') + glob.glob(f'{SRC}/valid/images/*'),
    'valid': other_end[gap:half],             # skip 3 frames after the train frames
    'test':  other_end[half + gap:],          # skip 3 frames after the valid frames
}

for split, imgs in plan.items():
    os.makedirs(f'{DST}/{split}/images', exist_ok=True)
    os.makedirs(f'{DST}/{split}/labels', exist_ok=True)
    for img in imgs:
        lbl = img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        shutil.copy(img, f'{DST}/{split}/images/')
        shutil.copy(lbl, f'{DST}/{split}/labels/')
    print(split, len(imgs), "images")

cfg = yaml.safe_load(open(f'{SRC}/data.yaml'))
cfg.update(path=DST, train='train/images', val='valid/images', test='test/images')
yaml.safe_dump(cfg, open(f'{DST}/data.yaml', 'w'))
DATA_V4 = f'{DST}/data.yaml'

In [ ]:
model = YOLO('yolo11n-pose.pt')
results = model.train(
    data=DATA_V4, epochs=300, imgsz=640, batch=16, fliplr=0.5,
    patience=0, project=f'{PROJECT_DIR}/runs', name='baseline_v4',
)

In [ ]:
v4 = YOLO(f'{PROJECT_DIR}/runs/baseline_v4/weights/last.pt')
te = v4.val(data=DATA_V4, split='test')
print("TEST pose mAP50:", round(te.pose.map50, 3), "| TEST box mAP50:", round(te.box.map50, 3))